**请介绍一下数据仓库的分层架构，以及你在项目中的实践**
  
在我之前的项目中，我们采用的是标准的五层架构：ODS、DIM、DWD、DWS和ADS。
简单来说，ODS层就是贴源层，源系统的数据原封不动地同步过来，不做太多处理。这样做的好处是，如果源系统数据结构变了，我们只需要改ODS的同步逻辑，不会影响下游。
DIM层是维度层，统一管理用户、商品、时间这些维度数据。
DWD层是我最关注的，这是明细事实层。我们有个很重要的原则——一个业务过程一张表。比如下单是一张表，支付是另一张表，不会混在一起。而且我们规定一个指标只能归属一张表，不允许同一指标在多张表里重复定义。之前遇到过这个问题，"支付金额"这个字段在两张表里都有，但命名不一样，一个叫payment_amt，一个叫pay_amount，下游用的时候经常搞混，所以需要在命名上区分它们
DWS层是汇总层，也是指标的统一出口。所有跨业务过程的指标组装都在这一层完成。比如说"用户下单后24小时内的支付率"，这个指标涉及下单和支付两个过程，就必须在DWS层做，DWD层不碰这种跨过程的指标。
ADS层就是看业务需求来灵活建设了
我们还有个硬性规定：禁止跨层取数。DWD必须从ODS取数，不能直接连源数据库。这样数据血缘很清晰，出了问题也容易排查。

**DWD层的事实表有哪几种？**
事务事实表：一行代表一个具体的业务操作，通常只插入，不更新，是最常用的
周期快照事实表：在固定时间间隔对业务状态进行快照，一行代表某个时间点（天/周/月）的汇总状态，示例：每日库存快照表、每月账户余额表、每天订单状态汇总表
累积快照事实表：跟踪业务实体的完整生命周期，记录关键里程碑，示例：订单生命周期表（包含下单时间、支付时间、发货时间、签收时间等）
无事实的事实表：只有维度外键，没有度量值，示例：用户签到表、活动参与记录表

**你的DWS层是怎么设计的？有什么原则？**
DWS层我们定位得很明确——它是数仓的指标统一出口，下游只能从这一层拿指标。
设计原则有几个：
第一，面向主题域建表，不面向报表建表。 比如说我们建"用户行为日汇总表"，而不是"XX报表数据表"。因为报表会变，但主题是相对稳定的。
第二，派生指标有四个要素：原子指标、修饰词、时间周期和聚合维度。 比如"近7日北京地区用户下单数"——"下单数"是原子指标，"北京地区"是修饰词，"近7日"是时间周期，"用户"是聚合维度。这四个要素组合起来才是一个完整的派生指标。
第三，不同时间粒度分表存储。 日表和月表分开建，不会混在一张表里。这样查询的时候分区裁剪更清晰，性能也更好。
第四，适度冗余。 有些高频筛选的维度我们会直接冗余到DWS表里，避免频繁JOIN。当然这个要权衡，不能为了冗余而冗余。
我们有个硬性规定：DWD层不做跨过程指标拼接。比如订单和售后的关联分析，必须在DWS层做，DWD层只负责把各自的事实表建好。

**维度建模中，什么是事实表的"粒度"？为什么它很重要？**
粒度是事实表中一行数据所代表的业务含义，简单说就是**"一行数据代表什么"**。
比如说订单事实表，如果粒度是"每个订单项"，那么一个订单如果有3个商品，就会有3行记录。如果粒度是"每个订单"，那不管有多少商品，都只有1行。
为什么重要？ 因为粒度决定了：
第一，数据能不能下钻分析。 如果粒度是"订单级别"，你就没法分析单个商品的销售情况了。反之，如果粒度是"订单项级别"，你可以向上聚合到订单级别，但反过来不行。所以我们的原则是DWD层尽量保持最细粒度，汇总的事交给DWS层做。
第二，指标计算是否正确。 比如计算GMV，如果粒度搞错了，可能会重复计算。我们之前就踩过坑，有个表粒度是"用户-商品-天"，但计算销售额的时候直接SUM，结果同一个订单被算了多次。
第三，存储和性能的权衡。 粒度越细，数据量越大，但灵活性越高。我们DWD层用细粒度，DWS层根据分析需求做适当的汇总。

**数据仓库的模型设计方法有哪些？你更倾向于哪种？**
第一种是范式建模（ER模型）。 就是传统数据库的第三范式设计，消除数据冗余。优点是数据一致性好，存储高效。缺点是查询复杂，JOIN太多，性能差。这种在传统数仓 like Teradata用得比较多。
第二种是维度建模。 就是Fact+Dimension的星型模型，也是我主要用的方法。优点是查询性能好，业务理解门槛低。缺点是有一定冗余，但随着存储成本降低，这个已经不是大问题了。
一般主流都是用的维度建模，原因是
业务友好：星型模型很直观，业务人员容易理解性能优秀，减少了JOIN，查询效率高
生态成熟：工具链完善，从建模到可视化的支持都很好
当然，我们也不是完全不用范式。ODS层其实就比较接近范式模型，保持和源系统一致。到了DWD层才转为维度模型。实际项目中往往是混合使用，不是非黑即白。

**数仓中如何处理缓慢变化维？你们的实践是什么？**
缓慢变化维（SCD）是维度建模里的经典问题。比如用户的收货地址变了、商品的价格调整了，这些维度属性不是经常变，但变了之后怎么处理？
常见的处理方式：
SCD Type 1（覆盖更新）：直接UPDATE，不保留历史。适用于纠错场景，比如用户手机号录错了，直接改过来就行，不需要保留错误版本。
SCD Type 2（新增行）：每次变化插入新记录，用生效时间、失效时间和当前标识来管理。适用于需要完整历史追溯的场景，比如分析用户在不同城市的消费行为变化。
SCD Type 3（新增列）：加一个"上次值"字段，只能保留最近一次变化。用得比较少，因为扩展性差。
我们的实践：
对于DIM层，大部分场景用SCD Type 2。但如果用Iceberg，我们有更简单的方案——时间旅行。 直接查询某个时间点的快照，这样就不需要维护复杂的SCD Type 2结构了，大大简化了ETL逻辑。
选择建议：
如果用的是传统Hive/MySQL，SCD Type 2是标准做法
如果用Iceberg等数据湖技术，优先考虑时间旅行
关键是看业务是否需要追溯历史，不需要的话Type 1就够了

**数仓建设中，遇到过最大的挑战是什么？怎么解决的？**
最大的挑战应该是指标口径不一致导致的数据信任危机。
当时的情况是，公司高管看报表，发现财务部门的数据、运营部门的数据、数仓的数据对不上。同样是"月销售额"，三个数字都不一样。高管直接质疑："你们的数据到底哪个是对的？"
这个问题很严重，因为一旦失去信任，数仓的价值就没了。
根本原因：
各部门自己定义指标，没有统一管理
数仓早期为了快速响应需求，没有严格执行规范
源系统本身数据质量就有问题（比如退款没及时同步）
解决过程：
第一步：止血。 先盘点所有对外报表，标注每个指标的来源和计算逻辑，找出差异点。
第二步：统一口径。 拉着业务、财务、运营开会，逐一确认每个指标的官方定义。这个过程很痛苦，因为各方都有自己的"道理"，最后只能高管拍板。
第三步：技术重构。
建立指标字典，明确每个指标的归属层和计算逻辑
DWD层严格遵循"一个指标一张表"
DWS层作为唯一出口，下游只能用DWS的数据
废弃那些不规范的临时表
第四步：建立流程。
新增指标必须走评审流程
指标变更要通知所有下游
定期做数据对账
第五步：重建信任。
主动和业务团队对齐数据
建立数据质量报告，定期发布
有问题及时沟通，不藏着
经验教训：
规范必须严格执行，不能因为赶工期就妥协
指标管理要趁早，越到后面越难改
沟通比技术重要，很多问题不是技术解决不了，而是各方利益不一致
透明度很重要，数据来源、计算逻辑要公开，接受监督



